In [16]:
#!/usr/bin/env python3
"""
erd_from_catalog_full.py
Deterministic ERD generator from PostgreSQL catalogs -> PlantUML (+ optional PNG).
"""

'\nerd_from_catalog_full.py\nDeterministic ERD generator from PostgreSQL catalogs -> PlantUML (+ optional PNG).\n'

In [17]:
import os
import psycopg2
import subprocess
import json

In [18]:
# -------- CONFIG (edit) --------
DB_HOST = os.environ.get("DB_HOST", "10.216.33.125")
DB_NAME = os.environ.get("DB_NAME", "simandb")
DB_USER = os.environ.get("DB_USER", "erd_adhoc")
DB_PASSWORD = os.environ.get("DB_PASSWORD", "erdG3nt!n9##")
DB_PORT = int(os.environ.get("DB_PORT", 5444))

OUTPUT_PUML = "erd_full_Catalog_1_simanstby1.puml"
OUTPUT_PNG = "erd_full_Catalog_1_simanstby1.png"
PLANTUML_JAR = os.environ.get("PLANTUML_JAR")  # optional path to plantuml.jar (java -jar)
INCLUDE_SCHEMAS = None  # None => include all non-system schemas; or set to ['public','myschema']
# --------------------------------

In [20]:
def connect():
    return psycopg2.connect(host=DB_HOST, database=DB_NAME, user=DB_USER, password=DB_PASSWORD, port=DB_PORT)

In [21]:

def list_user_schemas(cur):
    cur.execute("""
        SELECT nspname
        FROM pg_namespace
        WHERE nspname NOT IN ('pg_catalog','information_schema')
          AND nspname NOT LIKE 'pg_toast%'
          AND nspname NOT LIKE 'pg_temp%' -- tambahan agar yang dimasukkan hanya database terkait probis app
        ORDER BY nspname;
    """)
    schemas = [r[0] for r in cur.fetchall()]
    if INCLUDE_SCHEMAS:
        schemas = [s for s in schemas if s in INCLUDE_SCHEMAS]
    return schemas
    
def list_tables(cur, schemas):
    cur.execute("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_type='BASE TABLE'
          AND table_schema = ANY(%s)
        ORDER BY table_schema, table_name;
    """, (schemas,))
    return [(r[0], r[1]) for r in cur.fetchall()]

#def list_tables(cur, schemas):
#    cur.execute("""
#        SELECT table_schema, table_name, table_type
#        FROM information_schema.tables
#        WHERE table_type IN ('BASE TABLE','FOREIGN TABLE')
#          AND table_schema NOT IN ('pg_catalog','information_schema')
#        ORDER BY table_schema, table_name;
#    """)
    # return schema-qualified names so nothing collides
#    return [f"{row[0]}.{row[1]}" for row in cur.fetchall()]
    

def get_columns(cur, schema, table):
    cur.execute("""
        SELECT column_name, data_type, is_nullable, column_default
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position;
    """, (schema, table))
    return [dict(name=r[0], type=r[1], nullable=(r[2] == 'YES'), default=r[3]) for r in cur.fetchall()]

def get_primary_keys(cur, schema, table):
    cur.execute("""
    SELECT kcu.column_name
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
      ON tc.constraint_name = kcu.constraint_name
     AND tc.table_schema = kcu.table_schema
    WHERE tc.constraint_type = 'PRIMARY KEY' AND tc.table_schema=%s AND tc.table_name=%s
    ORDER BY kcu.ordinal_position;
    """, (schema, table))
    return [r[0] for r in cur.fetchall()]

def get_all_foreign_keys(cur, schemas):
    """
    Return list of foreign keys with:
    { constraint, schema_from, table_from, column_from, schema_to, table_to, column_to }
    """
    cur.execute("""
    SELECT
      con.conname AS constraint_name,
      n_from.nspname AS schema_from,
      cl_from.relname AS table_from,
      att_from.attname AS column_from,
      n_to.nspname AS schema_to,
      cl_to.relname AS table_to,
      att_to.attname AS column_to
    FROM pg_constraint con
    JOIN pg_class cl_from ON cl_from.oid = con.conrelid
    JOIN pg_namespace n_from ON n_from.oid = cl_from.relnamespace
    JOIN pg_class cl_to ON cl_to.oid = con.confrelid
    JOIN pg_namespace n_to ON n_to.oid = cl_to.relnamespace
    JOIN unnest(con.conkey) WITH ORDINALITY AS fk_cols(attnum, ord) ON true
    JOIN unnest(con.confkey)  WITH ORDINALITY AS pk_cols(attnum2, ord2) ON fk_cols.ord = pk_cols.ord2
    JOIN pg_attribute att_from ON att_from.attrelid = cl_from.oid AND att_from.attnum = fk_cols.attnum
    JOIN pg_attribute att_to   ON att_to.attrelid   = cl_to.oid   AND att_to.attnum   = pk_cols.attnum2
    WHERE con.contype = 'f'
      AND n_from.nspname = ANY(%s)
    ORDER BY constraint_name;
    """, (schemas,))
    rows = cur.fetchall()
    fks = []
    for r in rows:
        fks.append({
            "constraint": r[0],
            "schema_from": r[1],
            "table_from": r[2],
            "column_from": r[3],
            "schema_to": r[4],
            "table_to": r[5],
            "column_to": r[6]
        })
    return fks




In [22]:
def detect_join_table(schema, table, cols, pks, fks_for_table):
    """
    Heuristic: a join table often has:
      - PK composed of the two (or more) FK columns (or no other non-FK columns)
      - exactly two FKs to two different parent tables
    Return (is_join, join_fk_info)
    """
    if len(fks_for_table) < 2:
        return False, None
    # get set of fk columns
    fk_cols = {fk['column_from'] for fk in fks_for_table}
    non_fk_cols = [c for c in cols if c['name'] not in fk_cols]
    # If no non-FK columns OR only a serial id plus the two fks but PK is the two fks => join
    if len(non_fk_cols) == 0:
        # return pair of referenced tables
        parents = []
        for fk in fks_for_table:
            parents.append((fk['schema_to'], fk['table_to'], fk['column_to']))
        return True, parents
    # If primary key equals exactly the fk columns
    if set(pks) and set(pks) == fk_cols:
        parents = []
        for fk in fks_for_table:
            parents.append((fk['schema_to'], fk['table_to'], fk['column_to']))
        return True, parents
    return False, None


In [23]:
# --- heuristic FK inference when pg_constraint has none (for FDW tables) ---
def infer_foreign_keys_by_naming(tables_meta):
    """
    Heuristic: for each table, find columns ending with '_id' or '<tbl>_id' and match them
    to candidate target tables with that PK name or single-column PK 'id' and same type.
    Returns list of inferred fk dicts matching get_all_foreign_keys format.
    """
    inferred = []
    # build quick lookup: pk_name -> list of table keys having that pk column name
    pk_lookup = {}
    types_lookup = {}  # map table.key -> {colname: type}
    for tkey, meta in tables_meta.items():
        pkset = set(meta.get('pks', []))
        colmap = {c['name']: c['type'] for c in meta['columns']}
        types_lookup[tkey] = colmap
        for pk in pkset:
            pk_lookup.setdefault(pk, []).append(tkey)
        # also add 'id' if present
        if 'id' in colmap:
            pk_lookup.setdefault('id', []).append(tkey)

    for tkey, meta in tables_meta.items():
        for col in meta['columns']:
            cname = col['name']
            ctype = col['type']
            if cname.endswith('_id') or cname == 'id':
                base = cname[:-3] if cname.endswith('_id') else cname
                candidates = []
                # try direct match by base (table name)
                table_name_match = f"{meta['schema']}.{base}"
                if table_name_match in tables_meta:
                    candidates.append(table_name_match)
                # try pk lookup by column name
                matches = pk_lookup.get(cname, []) + pk_lookup.get(base, [])
                for m in matches:
                    if m not in candidates:
                        candidates.append(m)
                # check type compatibility
                for target in candidates:
                    target_col_type = types_lookup.get(target, {}).get(colname_from_pk := (cname if cname in types_lookup.get(target, {}) else 'id'), None)
                    # prefer same type or if target_col_type exists
                    if target_col_type and (target_col_type.split('(')[0] == ctype.split('(')[0]):
                        # build inferred fk
                        schema_to, table_to = target.split('.', 1)
                        inferred.append({
                            "constraint": f"inferred_{tkey.replace('.','_')}_{cname}_to_{table_to}",
                            "schema_from": meta['schema'],
                            "table_from": meta['table'],
                            "column_from": cname,
                            "schema_to": schema_to,
                            "table_to": table_to,
                            "column_to": colname_from_pk
                        })
                        break
    return inferred


In [24]:
def quote_ident(name):
    # safe PlantUML id: replace '.' with '__' and other unsafe chars
    return name.replace('.', '__').replace('-', '_')


In [25]:
def build_plantuml(tables_meta, fks):
    """
    tables_meta: dict of "schema.table" -> {schema, table, columns:[{name,type,...}], pks:[...]}
    fks: list of fk dicts (as returned by get_all_foreign_keys)
    """
    lines = []
    lines.append("@startuml")
    lines.append("skinparam classAttributeIconSize 0")
    lines.append("left to right direction")
    lines.append("hide circle")
    lines.append("scale 1.0\n")

    # Create entity blocks
    for key, meta in tables_meta.items():
        name_full = f"{meta['schema']}.{meta['table']}"
        idname = quote_ident(name_full)
        # lines.append(f'entity "{name_full}" as {idname} {{') - old version, belum cover semua
        # Check if we marked it as external in the main loop - tambahan cover possibility inter-relational schema
        color = "#E0E0E0" if meta.get('is_external') else "#FFFFFF"
        lines.append(f'entity "{name_full}" as {idname} {color} {{')
        # put PKs first
        pkset = set(meta.get('pks', []))
        # sort attributes: pk first, then others
        attrs_sorted = sorted(meta['columns'], key=lambda a: (0 if a['name'] in pkset else 1, a['name']))
        for a in attrs_sorted:
            pk_mark = " <<PK>>" if a['name'] in pkset else ""
            lines.append(f"  {a['name']} : {a['type']}{pk_mark}")
        lines.append("}")
        lines.append("")

    # Build a mapping of fks by from-table
    fks_by_from = {}
    for fk in fks:
        key_from = f"{fk['schema_from']}.{fk['table_from']}"
        fks_by_from.setdefault(key_from, []).append(fk)

    # Detect join tables to render many-to-many as direct relations (optional)
    join_tables = set()
    for tkey, meta in tables_meta.items():
        fks_for_table = fks_by_from.get(tkey, [])
        is_join, parents = detect_join_table(meta['schema'], meta['table'], meta['columns'], meta.get('pks', []), fks_for_table)
        if is_join:
            join_tables.add(tkey)

    # For each fk, generate arrow lines
    seen_rel = set()
    for fk in fks:
        from_full = f"{fk['schema_from']}.{fk['table_from']}"
        to_full = f"{fk['schema_to']}.{fk['table_to']}"
        # If the from table is a join table and we want to collapse many-to-many, skip drawing the join entity and create many-to-many line
        if from_full in join_tables:
            # gather parents (we will process pairs per join table later)
            continue
        # multiplicity inference: child (from) usually "0..*" or "1..*" and parent (to) usually "1" if referenced col is PK else "0..1"
        parent_is_pk = fk['column_to'] in (tables_meta.get(to_full, {}).get('pks') or [])
        child_mult = "\"0..*\""  # default: many children
        parent_mult = "\"1\"" if parent_is_pk else "\"0..1\""
        left = quote_ident(from_full)
        right = quote_ident(to_full)
        label = f"{fk['column_from']} -> {fk['column_to']}"
        key = (from_full, to_full, fk['column_from'])
        if key in seen_rel:
            continue
        seen_rel.add(key)
        # PlantUML format: left "0..*" --> "1" right : fk_col
        lines.append(f'{left} {child_mult} --> {parent_mult} {right} : {label}')

    # Handle join tables -> produce many-to-many between parent pairs
    for jt in join_tables:
        fk_list = fks_by_from.get(jt, [])
        if len(fk_list) < 2:
            continue
        # connect each parent to each other parent
        parent_ids = []
        parent_multiplicities = {}
        for fk in fk_list:
            p_full = f"{fk['schema_to']}.{fk['table_to']}"
            parent_ids.append(p_full)
            parent_is_pk = fk['column_to'] in (tables_meta.get(p_full, {}).get('pks') or [])
            parent_multiplicities[p_full] = "\"1\"" if parent_is_pk else "\"0..1\""
        # create pairwise many-to-many lines
        for i in range(len(parent_ids)):
            for j in range(i+1, len(parent_ids)):
                a = parent_ids[i]
                b = parent_ids[j]
                left = quote_ident(a)
                right = quote_ident(b)
                # many-to-many: both sides "0..*"
                lines.append(f'{left} "0..*" -- "0..*" {right} : many-to-many via {jt}')

    lines.append("@enduml")
    return "\n".join(lines)




In [26]:
def try_render_puml(puml_path, jar_path=None):
    png_path = os.path.splitext(puml_path)[0] + ".png"
    # try plantuml CLI
    try:
        #subprocess.run(["plantuml", "-tpng", puml_path], check=True)
        # Change extension to .svg
        svg_path = os.path.splitext(puml_path)[0] + ".svg"
        # Change flag to -tsvg
        subprocess.run(["plantuml", "-tsvg", puml_path], check=True)
        if os.path.exists(png_path):
            return png_path
    except FileNotFoundError:
        pass
    except subprocess.CalledProcessError as e:
        print("plantuml CLI failed:", e)
    # try java jar fallback
    if jar_path:
        try:
            subprocess.run(["java", "-jar", jar_path, "-tpng", puml_path], check=True)
            if os.path.exists(png_path):
                return png_path
        except FileNotFoundError:
            print("Java not found in PATH.")
        except subprocess.CalledProcessError as e:
            print("plantuml.jar failed:", e)
    return None



In [11]:
def main():
    conn = connect()
    cur = conn.cursor()
    try:
        schemas = list_user_schemas(cur)
        if not schemas:
            print("No user schemas found.")
            return
        print("Schemas considered:", schemas)
        tables = list_tables(cur, schemas)
        if not tables:
            print("No tables found.")
            return
        print(f"Found {len(tables)} tables (schema.table).")

        # collect metadata
        tables_meta = {}
        for schema, table in tables:
            key = f"{schema}.{table}"
            cols = get_columns(cur, schema, table)
            pks = get_primary_keys(cur, schema, table)
            tables_meta[key] = {'schema': schema, 'table': table, 'columns': cols, 'pks': pks}

fks = get_all_foreign_keys(cur, schemas)

### tambahan adaptasi atas modifikasi foreign tables ###
# If there are foreign tables and minimal or no fks for them, try inference
# Build a set of tables that are foreign according to pg_foreign_table
cur.execute("""
    SELECT nspname || '.' || relname AS fq
    FROM pg_foreign_table ft
    JOIN pg_class c ON c.oid = ft.ftrelid
    JOIN pg_namespace n ON n.oid = c.relnamespace
""")
foreign_tables = {r[0] for r in cur.fetchall()}

# Only infer for tables that lack fk entries
if foreign_tables:
    # decide whether to infer: if any foreign table has zero FK rows in fks_by_from
    fks_by_from = {}
    for fk in fks:
        fks_by_from.setdefault(f"{fk['schema_from']}.{fk['table_from']}", []).append(fk)
    need_infer = any(ft not in fks_by_from for ft in foreign_tables)
    if need_infer:
        inferred = infer_foreign_keys_by_naming(tables_meta)
        if inferred:
            print(f"Added {len(inferred)} inferred FK(s) for foreign tables (heuristic).")
            fks.extend(inferred)

        # Build PlantUML
        puml = build_plantuml(tables_meta, fks)
        with open(OUTPUT_PUML, "w", encoding="utf-8") as fh:
            fh.write(puml)
        print("Saved PUML to", OUTPUT_PUML)
### tambahan adaptasi atas modifikasi foreign tables ###
        
        # Optionally render
        png = try_render_puml(OUTPUT_PUML, jar_path=PLANTUML_JAR)
        if png:
            print("Rendered PNG:", png)
        else:
            print("No renderer found or rendering failed. You can render erd_full.puml locally with PlantUML.")

        # Optional: save JSON metadata for inspection
        with open("erd_catalog_meta.json", "w", encoding="utf-8") as fh:
            json.dump({'tables': tables_meta, 'fks': fks}, fh, indent=2, default=str)
        print("Saved catalog metadata to erd_catalog_meta.json")

    finally:
        cur.close()
        conn.close()

if __name__ == "__main__":
    main()

SyntaxError: expected 'except' or 'finally' block (1886089109.py, line 24)

In [54]:
def main():
    conn = connect()
    cur = conn.cursor()
    try:
        schemas = list_user_schemas(cur)
        if not schemas:
            print("No user schemas found.")
            return
        print("Schemas considered:", schemas)

        tables = list_tables(cur, schemas)
        if not tables:
            print("No tables found.")
            return
        print(f"Found {len(tables)} tables (schema.table).")

        # --- collect metadata ---
        tables_meta = {}
        for schema, table in tables:
            key = f"{schema}.{table}"
            cols = get_columns(cur, schema, table)
            pks = get_primary_keys(cur, schema, table)
            tables_meta[key] = {
                'schema': schema,
                'table': table,
                'columns': cols,
                'pks': pks
            }

        # --- get local FK constraints ---
        fks = get_all_foreign_keys(cur, schemas)

        # =====================================================================
        #      FDW ENHANCEMENT SECTION (FOREIGN TABLE DETECTION + INFERENCE)
        # =====================================================================
        cur.execute("""
            SELECT nspname || '.' || relname AS fq
            FROM pg_foreign_table ft
            JOIN pg_class c ON c.oid = ft.ftrelid
            JOIN pg_namespace n ON n.oid = c.relnamespace
        """)
        foreign_tables = {r[0] for r in cur.fetchall()}

        if foreign_tables:
            print("Foreign tables detected:", foreign_tables)

            # Map FK rows grouped by table_from
            fks_by_from = {}
            for fk in fks:
                fks_by_from.setdefault(f"{fk['schema_from']}.{fk['table_from']}", []).append(fk)

            need_infer = any(ft not in fks_by_from for ft in foreign_tables)

            if need_infer:
                print("Some foreign tables have no FK metadata locally. Running inference...")
                inferred = infer_foreign_keys_by_naming(tables_meta)

                if inferred:
                    print(f"Added {len(inferred)} inferred FK(s) for foreign tables.")
                    fks.extend(inferred)
                else:
                    print("No inferred FK relationships could be determined.")

        # =====================================================================
        #                         Build PlantUML
        # =====================================================================
        puml = build_plantuml(tables_meta, fks)
        with open(OUTPUT_PUML, "w", encoding="utf-8") as fh:
            fh.write(puml)
        print("Saved PUML to", OUTPUT_PUML)

        # --- Render PNG ---
        png = try_render_puml(OUTPUT_PUML, jar_path=PLANTUML_JAR)
        if png:
            print("Rendered PNG:", png)
        else:
            print("Rendering skipped or PlantUML not installed.")

        # --- Save catalog metadata JSON ---
        with open("erd_catalog_meta.json", "w", encoding="utf-8") as fh:
            json.dump({'tables': tables_meta, 'fks': fks}, fh, indent=2, default=str)
        print("Saved catalog metadata to erd_catalog_meta.json")

    finally:
        cur.close()
        conn.close()


if __name__ == "__main__":
    main()


Schemas considered: ['aq$_agent', 'aq$_descriptor', 'aq$_reg_info', 'asuransi', 'auth', 'bmnidle', 'dbms_aq', 'dbms_aqadm', 'evaluasi', 'inventarisasi', 'kelola', 'migration', 'msg_prop_t', 'pg_temp_10', 'pg_temp_100', 'pg_temp_1000', 'pg_temp_1001', 'pg_temp_1002', 'pg_temp_1003', 'pg_temp_1004', 'pg_temp_1005', 'pg_temp_1006', 'pg_temp_1007', 'pg_temp_1008', 'pg_temp_1009', 'pg_temp_101', 'pg_temp_1010', 'pg_temp_1011', 'pg_temp_1012', 'pg_temp_1013', 'pg_temp_1014', 'pg_temp_1015', 'pg_temp_1016', 'pg_temp_1017', 'pg_temp_1018', 'pg_temp_1019', 'pg_temp_102', 'pg_temp_1020', 'pg_temp_1021', 'pg_temp_1022', 'pg_temp_1023', 'pg_temp_1024', 'pg_temp_1025', 'pg_temp_1026', 'pg_temp_1027', 'pg_temp_1028', 'pg_temp_1029', 'pg_temp_103', 'pg_temp_1030', 'pg_temp_1031', 'pg_temp_1032', 'pg_temp_1033', 'pg_temp_1034', 'pg_temp_1035', 'pg_temp_1036', 'pg_temp_1037', 'pg_temp_1038', 'pg_temp_1039', 'pg_temp_104', 'pg_temp_1040', 'pg_temp_1041', 'pg_temp_1042', 'pg_temp_1043', 'pg_temp_1044', '

In [15]:
def main():
    conn = connect()
    cur = conn.cursor()
    try:
        # 1. Get List of Schemas
        # (Make sure list_user_schemas is updated to ignore 'pg_temp%'!)
        schemas = list_user_schemas(cur)
        if not schemas:
            print("No user schemas found.")
            return
        print(f"Found {len(schemas)} schemas. Starting generation per schema...")

        # 2. LOOP: Process one schema at a time
        for schema in schemas:
            print(f"\n--- Processing Schema: {schema} ---")

            # Get tables for ONLY this schema
            tables = list_tables(cur, [schema])
            if not tables:
                print(f"  No tables found in {schema}. Skipping.")
                continue

            # Collect metadata for this schema
            tables_meta = {}
            for s, t in tables:
                key = f"{s}.{t}"
                cols = get_columns(cur, s, t)
                pks = get_primary_keys(cur, s, t)
                tables_meta[key] = {
                    'schema': s, 
                    'table': t, 
                    'columns': cols, 
                    'pks': pks
                }

            # Get Foreign Keys originating from this schema
            fks = get_all_foreign_keys(cur, [schema])

            # 3. Generate PUML for this schema
            puml_content = build_plantuml(tables_meta, fks)
            
            # Dynamic filename based on schema name
            puml_filename = f"erd_{schema}.puml"
            
            with open(puml_filename, "w", encoding="utf-8") as fh:
                fh.write(puml_content)
            print(f"  Saved PUML: {puml_filename}")

            # 4. Render to SVG (Scalable Vector Graphics)
            # SVG handles large diagrams much better than PNG
            svg_filename = f"erd_{schema}.svg"
            
            # Check if PlantUML is available
            try:
                # We use -tsvg flag here
                cmd = ["plantuml", "-tsvg", puml_filename]
                
                # If using a JAR file, adjust command:
                if PLANTUML_JAR:
                     cmd = ["java", "-jar", PLANTUML_JAR, "-tsvg", puml_filename]

                subprocess.run(cmd, check=True)
                
                if os.path.exists(svg_filename):
                    print(f"  Rendered SVG: {svg_filename}")
            except FileNotFoundError:
                print("  PlantUML executable not found (skipped rendering).")
            except subprocess.CalledProcessError as e:
                print(f"  Rendering failed for {schema}: {e}")
            except Exception as e:
                print(f"  Error rendering {schema}: {e}")

    finally:
        cur.close()
        conn.close()

if __name__ == "__main__":
    main()

Found 21 schemas. Starting generation per schema...

--- Processing Schema: aq$_agent ---
  No tables found in aq$_agent. Skipping.

--- Processing Schema: aq$_descriptor ---
  No tables found in aq$_descriptor. Skipping.

--- Processing Schema: aq$_reg_info ---
  No tables found in aq$_reg_info. Skipping.

--- Processing Schema: asuransi ---
  Saved PUML: erd_asuransi.puml

--- Processing Schema: auth ---
  Saved PUML: erd_auth.puml

--- Processing Schema: bmnidle ---
  Saved PUML: erd_bmnidle.puml

--- Processing Schema: dbms_aq ---
  No tables found in dbms_aq. Skipping.

--- Processing Schema: dbms_aqadm ---
  No tables found in dbms_aqadm. Skipping.

--- Processing Schema: evaluasi ---
  Saved PUML: erd_evaluasi.puml

--- Processing Schema: inventarisasi ---
  Saved PUML: erd_inventarisasi.puml

--- Processing Schema: kelola ---
  Saved PUML: erd_kelola.puml

--- Processing Schema: migration ---
  Saved PUML: erd_migration.puml

--- Processing Schema: msg_prop_t ---
  No tables fo

In [27]:
### SCRIPT EXEC VERSI 3 UNTUK COVER JUGA POSSIBILITY REFERENTIAL INTEGRITY ANTAR SCHEMA DI DB SIMAN V2 ###
def main():
    conn = connect()
    cur = conn.cursor()
    try:
        # 1. Get clean list of schemas (filtering out pg_temp and system schemas)
        # Ensure list_user_schemas has the filter: AND nspname NOT LIKE 'pg_temp%'
        schemas = list_user_schemas(cur)
        if not schemas:
            print("No user schemas found.")
            return
        
        print(f"Found {len(schemas)} schemas. Generating Inter-Schema Contextual Diagrams...")

        # ---------------------------------------------------------
        # LOOP: Generate one diagram per schema, BUT include its external parents
        # ---------------------------------------------------------
        for current_schema in schemas:
            print(f"\n--- Processing Schema: {current_schema} ---")

            # A. Get Local Tables
            local_tables = list_tables(cur, [current_schema])
            if not local_tables:
                print(f"  No tables found in {current_schema}. Skipping.")
                continue

            # B. Initialize Metadata with Local Tables
            tables_meta = {}
            for s, t in local_tables:
                key = f"{s}.{t}"
                cols = get_columns(cur, s, t)
                pks = get_primary_keys(cur, s, t)
                tables_meta[key] = {
                    'schema': s, 
                    'table': t, 
                    'columns': cols, 
                    'pks': pks,
                    'is_external': False  # Mark as local
                }

            # C. Get Foreign Keys originating from CURRENT schema
            # This returns links where schema_from == current_schema
            fks = get_all_foreign_keys(cur, [current_schema])

            # D. IDENTIFY & FETCH EXTERNAL PARENTS
            # Check if any FK points to a schema that is NOT the current one
            external_tables_to_fetch = set()
            for fk in fks:
                target_schema = fk['schema_to']
                target_table = fk['table_to']
                
                # If the target is in a different schema, we need to fetch its metadata
                # so PlantUML can draw the "To" side of the arrow.
                if target_schema != current_schema:
                    external_tables_to_fetch.add((target_schema, target_table))

            if external_tables_to_fetch:
                print(f"  found {len(external_tables_to_fetch)} external dependencies (inter-schema links).")

            # Fetch metadata for these specific external tables
            for ext_s, ext_t in external_tables_to_fetch:
                key = f"{ext_s}.{ext_t}"
                # Avoid duplicates if multiple tables point to the same external parent
                if key not in tables_meta:
                    cols = get_columns(cur, ext_s, ext_t)
                    pks = get_primary_keys(cur, ext_s, ext_t)
                    tables_meta[key] = {
                        'schema': ext_s, 
                        'table': ext_t, 
                        'columns': cols, 
                        'pks': pks,
                        'is_external': True # Mark as external (optional: could use to color differently)
                    }

            # E. Generate PlantUML
            # tables_meta now contains: All Local Tables + Any External Tables that are referenced
            puml_content = build_plantuml(tables_meta, fks)
            
            # F. Save and Render
            puml_filename = f"erd_inter_{current_schema}.puml"
            svg_filename = f"erd_inter_{current_schema}.svg"

            with open(puml_filename, "w", encoding="utf-8") as fh:
                fh.write(puml_content)
            
            # Try Render to SVG
            try:
                cmd = ["plantuml", "-tsvg", puml_filename]
                if PLANTUML_JAR:
                     cmd = ["java", "-jar", PLANTUML_JAR, "-tsvg", puml_filename]
                
                subprocess.run(cmd, check=True)
                if os.path.exists(svg_filename):
                    print(f"  Generated: {svg_filename}")
            except Exception as e:
                print(f"  Render failed: {e}")

    finally:
        cur.close()
        conn.close()

if __name__ == "__main__":
    main()

Found 21 schemas. Generating Inter-Schema Contextual Diagrams...

--- Processing Schema: aq$_agent ---
  No tables found in aq$_agent. Skipping.

--- Processing Schema: aq$_descriptor ---
  No tables found in aq$_descriptor. Skipping.

--- Processing Schema: aq$_reg_info ---
  No tables found in aq$_reg_info. Skipping.

--- Processing Schema: asuransi ---

--- Processing Schema: auth ---
  found 1 external dependencies (inter-schema links).

--- Processing Schema: bmnidle ---

--- Processing Schema: dbms_aq ---
  No tables found in dbms_aq. Skipping.

--- Processing Schema: dbms_aqadm ---
  No tables found in dbms_aqadm. Skipping.

--- Processing Schema: evaluasi ---

--- Processing Schema: inventarisasi ---

--- Processing Schema: kelola ---
  found 1 external dependencies (inter-schema links).

--- Processing Schema: migration ---
  found 2 external dependencies (inter-schema links).

--- Processing Schema: msg_prop_t ---
  No tables found in msg_prop_t. Skipping.

--- Processing Sch